# yt2tg — YouTube → Telegram Uploader

**Resume-safe**: state is saved after each upload. If Colab disconnects, just re-run — already-uploaded videos are skipped.

---
### First time only:
1. Run **Cell 1** (install + clone)
2. Run **Cell 2** (config — fill in your credentials)
3. Upload your `cookies.txt` when prompted
4. Run **Cell 3** onwards

### Resuming after Colab disconnect:
- Re-run **Cell 1** (re-installs deps, re-clones if needed)
- Re-run **Cell 2** (re-applies config)
- Run **Cell 3** again — completed videos are skipped automatically

In [ ]:
# ═══ Cell 1: Install dependencies ═══
import os, subprocess, sys

REPO_DIR = '/content/yt2tg'

# Clone or update repo
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/sudo-change/yt2tg.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# Install dependencies
!pip install -q pyrogram tgcrypto yt-dlp

# Verify
!yt-dlp --version
print('✓ Setup complete')

In [ ]:
# ═══ Cell 2: Configuration ═══
# Fill in your credentials here. This writes config.json into the repo folder.
import json, os

REPO_DIR = '/content/yt2tg'

# ─── FILL THESE IN ───────────────────────────────────────────────
BOT_TOKEN    = ""          # From @BotFather
API_ID       = ""          # From https://my.telegram.org
API_HASH     = ""          # From https://my.telegram.org
GROUP_CHAT_ID = 0          # Your supergroup chat ID (negative number, e.g. -1001234567890)
# ─────────────────────────────────────────────────────────────────

config = {
    "bot_token":       BOT_TOKEN,
    "api_id":          API_ID,
    "api_hash":        API_HASH,
    "group_chat_id":   GROUP_CHAT_ID,
    "channel_mappings": {}
}

config_path = os.path.join(REPO_DIR, 'config.json')

# Preserve existing channel_mappings if config already exists
if os.path.exists(config_path):
    with open(config_path) as f:
        existing = json.load(f)
    config['channel_mappings'] = existing.get('channel_mappings', {})

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f'✓ Config written to {config_path}')
print(f'  Bot: {BOT_TOKEN[:10]}...')
print(f'  Group: {GROUP_CHAT_ID}')

In [ ]:
# ═══ Cell 3: Upload cookies.txt ═══
# Export cookies from your browser (while logged into YouTube) and upload here.
# Re-run this cell whenever your cookies expire.
from google.colab import files
import shutil, os

REPO_DIR = '/content/yt2tg'

uploaded = files.upload()
if uploaded:
    fname = list(uploaded.keys())[0]
    dest  = os.path.join(REPO_DIR, 'cookies.txt')
    shutil.move(fname, dest)
    print(f'✓ Cookies saved to {dest}')
else:
    print('[WARN] No file uploaded — membership/private videos will fail')

In [ ]:
# ═══ Cell 4: Preview (dry-run) ═══
# See what videos will be processed WITHOUT downloading anything.
# Change the URLs and flags below, then run.

REPO_DIR = '/content/yt2tg'

# Examples:
#   Single video:      'https://youtube.com/watch?v=abc123'
#   Full channel:      'https://youtube.com/@NahamSec'
#   Members-only:  -m  'https://youtube.com/@NahamSec'

URLS  = [
    'https://youtube.com/@NahamSec',
]
FLAGS = '-m'    # -m for membership only, '' for all

url_args = ' '.join(f'"{u}"' for u in URLS)
!cd {REPO_DIR} && python yt2tg.py --dry-run {FLAGS} {url_args}

In [ ]:
# ═══ Cell 5: Download & Upload ═══
# -y skips confirmation prompts (required for unattended Colab runs)
# If Colab disconnects mid-run: just re-run this cell. Already uploaded videos are skipped.

REPO_DIR = '/content/yt2tg'

URLS  = [
    'https://youtube.com/@NahamSec',
]
FLAGS = '-m -y'    # -m membership only, -y skip confirmation

url_args = ' '.join(f'"{u}"' for u in URLS)
!cd {REPO_DIR} && python yt2tg.py {FLAGS} {url_args}

In [ ]:
# ═══ Cell 6: Check progress / state ═══
import json, os

REPO_DIR = '/content/yt2tg'

state_path = os.path.join(REPO_DIR, 'state.json')
if os.path.exists(state_path):
    with open(state_path) as f:
        state = json.load(f)
    print(f"✓ Completed : {len(state.get('completed', []))} videos")
    print(f"✗ Failed    : {len(state.get('failed', {}))} videos")
    if state.get('failed'):
        print('\nFailed videos:')
        for vid_id, reason in state['failed'].items():
            print(f'  {vid_id}: {reason[:80]}')
else:
    print('No state.json yet — nothing has been processed')

In [ ]:
# ═══ Cell 7: Reset state for specific videos (optional) ═══
# If you want to re-upload specific videos, remove them from state.json
import json, os

REPO_DIR = '/content/yt2tg'
state_path = os.path.join(REPO_DIR, 'state.json')

VIDEO_IDS_TO_RETRY = [
    # 'abc123',
    # 'xyz789',
]

if VIDEO_IDS_TO_RETRY and os.path.exists(state_path):
    with open(state_path) as f:
        state = json.load(f)
    state['completed'] = [v for v in state.get('completed', []) if v not in VIDEO_IDS_TO_RETRY]
    for v in VIDEO_IDS_TO_RETRY:
        state.get('failed', {}).pop(v, None)
    with open(state_path, 'w') as f:
        json.dump(state, f, indent=2)
    print(f'✓ Removed {len(VIDEO_IDS_TO_RETRY)} video(s) from state — will re-process on next run')
else:
    print('No video IDs specified or no state file found')